# B2 — WavLM-base + CTC Fine-tuning on TORGO (Fixed v2)

**Key fixes from original:**
1. Proper CTC decoding (collapse repeated tokens + remove blanks)
2. Higher learning rate: 3e-4 (standard for WavLM/wav2vec2 CTC fine-tuning)
3. Let Trainer handle optimizer/scheduler (custom optimizer caused LR decay bug)
4. 40 epochs with early stopping patience 8
5. Effective batch size 16
6. Sanity check callback to monitor predictions each epoch

Split:
- Test:  F01 (severe), F04 (mild), FC01 (control), M05 (moderate)
- Val:   ~10% utterances sampled from each training speaker (speaker-mixed)
- Train: all remaining speakers, control downsampled to ~800 utterances

In [1]:
import os

# ── FIX: Force single GPU BEFORE any torch import ──
# Must be set before importing torch, otherwise DataParallel uses both GPUs and OOMs
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ── Cache everything in /kaggle/temp so /kaggle/working stays under 20GB quota ──
os.environ["HF_HOME"]             = "/kaggle/temp/hf"
os.environ["HF_DATASETS_CACHE"]   = "/kaggle/temp/hf/datasets"
os.environ["HF_HUB_CACHE"]        = "/kaggle/temp/hf/hub"
os.environ["TRANSFORMERS_CACHE"]  = "/kaggle/temp/hf/transformers"
os.environ["XDG_CACHE_HOME"]      = "/kaggle/temp/.cache"

for p in [
    os.environ["HF_HOME"],
    os.environ["HF_DATASETS_CACHE"],
    os.environ["HF_HUB_CACHE"],
    os.environ["TRANSFORMERS_CACHE"],
    os.environ["XDG_CACHE_HOME"],
]:
    os.makedirs(p, exist_ok=True)

# Clean any old working-dir cache files
!rm -rf /kaggle/working/.cache /kaggle/working/hf /kaggle/working/huggingface
print("Cache dirs ready.")

Cache dirs ready.


In [2]:
# ── Install dependencies ──
!pip -q install -U transformers datasets accelerate evaluate jiwer soundfile packaging

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 72.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 96.2 MB/s eta 0:00:00:00:01


In [3]:
# ── Imports ──
import numpy as np
import pandas as pd
import torch
import evaluate
import random
import json
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union
from itertools import groupby

from datasets import load_dataset, Dataset, DatasetDict, load_from_disk
from transformers import (
    WavLMForCTC,
    Wav2Vec2Processor,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
)
import transformers
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

transformers: 5.5.0
torch: 2.10.0+cu128
GPU available: True
GPU name: Tesla T4


## Step 1 — Load TORGO and Build the Split

Identical data source as your Whisper baseline (`abnerh/TORGO-database`).
Speaker extraction logic is the same.
New additions: speaker-mixed validation, control downsampling.

In [4]:
# ── 1a) Load raw dataset ──
ds = load_dataset("abnerh/TORGO-database", cache_dir=os.environ["HF_DATASETS_CACHE"])
df = ds["train"].to_pandas()

# Extract speaker ID from audio path
df["audio_path"] = df["audio"].apply(lambda x: x["path"])
df["speaker"]    = df["audio_path"].apply(lambda p: str(p).split("_")[0])

print("Total utterances:", len(df))
print("Unique speakers:", sorted(df["speaker"].unique()))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/370M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/397M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/356M [00:00<?, ?B/s]

data/train-00003-of-00004.parquet:   0%|          | 0.00/441M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16552 [00:00<?, ? examples/s]

Total utterances: 16552
Unique speakers: ['F01', 'F03', 'F04', 'FC01', 'FC02', 'FC03', 'M01', 'M02', 'M03', 'M04', 'M05', 'MC01', 'MC02', 'MC03', 'MC04']


In [5]:
# ── 1b) Severity mapping ──
severity_mapping = {
    "F01": "severe",
    "M01": "severe",
    "M02": "severe",
    "M04": "severe",
    "M05": "moderate",
    "F03": "moderate",
    "F04": "mild",
    "M03": "mild",
    "FC01": "control",
    "FC02": "control",
    "FC03": "control",
    "MC01": "control",
    "MC02": "control",
    "MC03": "control",
    "MC04": "control",
}

df["Category"] = df["speaker"].map(severity_mapping)
print(df["Category"].value_counts())

Category
control     10978
severe       2417
moderate     1674
mild         1483
Name: count, dtype: int64


In [6]:
# ── 1c) Assign TEST speakers, then create speaker-mixed validation split ──
TEST_SPEAKERS = {"F01", "F04", "FC01", "M05"}

VAL_FRAC_PER_SPEAKER = 0.10
VAL_MAX_PER_SPEAKER  = 75
VAL_EXCLUDE_CATS     = {"mild"}
RANDOM_SEED          = 42

df["split"] = "train"
df.loc[df["speaker"].isin(TEST_SPEAKERS), "split"] = "test"

train_pool = df[df["split"] == "train"].copy()
val_candidates = train_pool[~train_pool["Category"].isin(VAL_EXCLUDE_CATS)].copy()

val_parts = []
for speaker, spk_df in val_candidates.groupby("speaker"):
    n_val = max(1, int(round(len(spk_df) * VAL_FRAC_PER_SPEAKER)))
    n_val = min(n_val, VAL_MAX_PER_SPEAKER, len(spk_df))
    val_parts.append(spk_df.sample(n=n_val, random_state=RANDOM_SEED))

val_df = pd.concat(val_parts).sort_index().copy()
df.loc[val_df.index, "split"] = "val"

print("\nSplit counts:")
print(df["split"].value_counts())
print("\nSplit × Category breakdown:")
print(pd.crosstab(df["split"], df["Category"]))
print("\nValidation speakers:")
print(sorted(df.loc[df["split"] == "val", "speaker"].unique()))


Split counts:
split
train    14013
test      1798
val        741
Name: count, dtype: int64

Split × Category breakdown:
Category  control  mild  moderate  severe
split                                    
test          302   673       587     236
train       10226   810      1012    1965
val           450     0        75     216

Validation speakers:
['F03', 'FC02', 'FC03', 'M01', 'M02', 'M04', 'MC01', 'MC02', 'MC03', 'MC04']


In [8]:
# ── 1d) Downsample control utterances in TRAINING only ──
CONTROL_TRAIN_TARGET = 1500

train_df = df[df["split"] == "train"].copy()
train_control = train_df[train_df["Category"] == "control"]
train_dysarth = train_df[train_df["Category"] != "control"]

train_control_sampled = train_control.sample(
    n=min(CONTROL_TRAIN_TARGET, len(train_control)),
    random_state=RANDOM_SEED
)

train_df_final = pd.concat([train_dysarth, train_control_sampled]).sample(
    frac=1, random_state=RANDOM_SEED
)

val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

print("Final training split:")
print(train_df_final["Category"].value_counts())
print(f"\nTotal train: {len(train_df_final)}")
print(f"Total val:   {len(val_df)}")
print(f"Total test:  {len(test_df)}")
print("\nTraining speakers:", sorted(train_df_final["speaker"].unique()))
print("Validation speakers:", sorted(val_df["speaker"].unique()))
print("Test speakers:", sorted(test_df["speaker"].unique()))

Final training split:
Category
severe      1965
control     1500
moderate    1012
mild         810
Name: count, dtype: int64

Total train: 5287
Total val:   741
Total test:  1798

Training speakers: ['F03', 'FC02', 'FC03', 'M01', 'M02', 'M03', 'M04', 'MC01', 'MC02', 'MC03', 'MC04']
Validation speakers: ['F03', 'FC02', 'FC03', 'M01', 'M02', 'M04', 'MC01', 'MC02', 'MC03', 'MC04']
Test speakers: ['F01', 'F04', 'FC01', 'M05']


In [9]:
# ── 1e) Convert back to HuggingFace datasets ──
hf_full = ds["train"].add_column("speaker", df["speaker"].tolist())
hf_full = hf_full.add_column("Category", df["Category"].tolist())
hf_full = hf_full.add_column("split", df["split"].tolist())

train_indices = train_df_final.index.tolist()
val_indices   = val_df.index.tolist()
test_indices  = test_df.index.tolist()

assert len(set(train_indices) & set(val_indices)) == 0
assert len(set(train_indices) & set(test_indices)) == 0
assert len(set(val_indices) & set(test_indices)) == 0

hf_dict = DatasetDict({
    "train": hf_full.select(train_indices),
    "val":   hf_full.select(val_indices),
    "test":  hf_full.select(test_indices),
})

SPLIT_SAVE_PATH = "/kaggle/working/torgo_b2_split"
hf_dict.save_to_disk(SPLIT_SAVE_PATH)
print("Saved split to:", SPLIT_SAVE_PATH)
print(hf_dict)

Saving the dataset (0/2 shards):   0%|          | 0/5287 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/741 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1798 [00:00<?, ? examples/s]

Saved split to: /kaggle/working/torgo_b2_split
DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription', 'speech_status', 'gender', 'duration', 'speaker', 'Category', 'split'],
        num_rows: 5287
    })
    val: Dataset({
        features: ['audio', 'transcription', 'speech_status', 'gender', 'duration', 'speaker', 'Category', 'split'],
        num_rows: 741
    })
    test: Dataset({
        features: ['audio', 'transcription', 'speech_status', 'gender', 'duration', 'speaker', 'Category', 'split'],
        num_rows: 1798
    })
})


## Step 2 — Build the CTC Vocabulary and Processor

- Character-level vocabulary: one token per character
- `|` = word delimiter (space), `[PAD]` = CTC blank, `[UNK]` = unknown
- Feature extractor normalises raw waveforms (zero mean, unit variance)

In [10]:
# ── 2a/2b/2c) Build character vocabulary, tokenizer, feature extractor, processor ──
import json
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

all_transcriptions = df["transcription"].dropna().tolist()
all_transcriptions = [t.lower().strip() for t in all_transcriptions]

all_chars = set()
for t in all_transcriptions:
    all_chars.update(list(t))

allowed = set("abcdefghijklmnopqrstuvwxyz '")
vocab_chars = sorted(all_chars.intersection(allowed))

print("Characters in vocabulary:", vocab_chars)
print("Vocab size (before special tokens):", len(vocab_chars))

# Build vocab
vocab_dict = {char: idx for idx, char in enumerate(vocab_chars)}

# Replace actual space with word delimiter token |
if " " in vocab_dict:
    space_id = vocab_dict[" "]
    del vocab_dict[" "]
    vocab_dict["|"] = space_id

# Add special tokens
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

VOCAB_PATH = "/kaggle/working/vocab.json"
with open(VOCAB_PATH, "w") as f:
    json.dump(vocab_dict, f, indent=2)

print("Vocab saved to:", VOCAB_PATH)
print("Total vocab size in vocab.json:", len(vocab_dict))
print(vocab_dict)

# Tokenizer
tokenizer = Wav2Vec2CTCTokenizer(
    VOCAB_PATH,
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

# Feature extractor
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

# Processor
processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

PROCESSOR_SAVE_PATH = "/kaggle/working/wavlm_processor"
processor.save_pretrained(PROCESSOR_SAVE_PATH)

print("Processor saved to:", PROCESSOR_SAVE_PATH)
print("len(tokenizer):", len(processor.tokenizer))
print("tokenizer.vocab_size:", processor.tokenizer.vocab_size)
print("pad token id:", processor.tokenizer.pad_token_id)
print("word delimiter token:", processor.tokenizer.word_delimiter_token)

Characters in vocabulary: [' ', "'", 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Vocab size (before special tokens): 28
Vocab saved to: /kaggle/working/vocab.json
Total vocab size in vocab.json: 30
{"'": 1, 'a': 2, 'b': 3, 'c': 4, 'd': 5, 'e': 6, 'f': 7, 'g': 8, 'h': 9, 'i': 10, 'j': 11, 'k': 12, 'l': 13, 'm': 14, 'n': 15, 'o': 16, 'p': 17, 'q': 18, 'r': 19, 's': 20, 't': 21, 'u': 22, 'v': 23, 'w': 24, 'x': 25, 'y': 26, 'z': 27, '|': 0, '[UNK]': 28, '[PAD]': 29}
Processor saved to: /kaggle/working/wavlm_processor
len(tokenizer): 32
tokenizer.vocab_size: 30
pad token id: 29
word delimiter token: |


In [ ]:
# merged into previous cell

In [ ]:
# merged into previous cell

## Step 3 — Preprocessing

WavLM-CTC pipeline:
```
audio array → feature_extractor → normalised raw waveform (variable length)
text → tokenizer → character IDs  e.g. 'hello' → [7,4,11,11,14]
```

In [11]:
# ── Filter out very long audio clips (>15s at 16kHz = 240,000 samples) ──
# Long clips cause huge attention matrices and OOM in WavLM's relative position bias
MAX_AUDIO_SAMPLES = 240_000  # 15 seconds

def filter_long_audio(example):
    return len(example["audio"]["array"]) <= MAX_AUDIO_SAMPLES

before = {k: len(hf_dict[k]) for k in hf_dict}
hf_dict_filtered = hf_dict.filter(filter_long_audio, num_proc=1)
after = {k: len(hf_dict_filtered[k]) for k in hf_dict_filtered}

print("Audio length filtering (max 15s):")
for split in before:
    print(f"  {split}: {before[split]} -> {after[split]} ({before[split]-after[split]} removed)")

hf_dict = hf_dict_filtered

Filter (num_proc=1):   0%|          | 0/5287 [00:00<?, ? examples/s]

Filter (num_proc=1):   0%|          | 0/741 [00:00<?, ? examples/s]

Filter (num_proc=1):   0%|          | 0/1798 [00:00<?, ? examples/s]

Audio length filtering (max 15s):
  train: 5287 -> 5259 (28 removed)
  val: 741 -> 736 (5 removed)
  test: 1798 -> 1786 (12 removed)


In [12]:
# ── Preprocessing ──
def prepare_example(batch):
    audio = batch["audio"]

    # Audio inputs
    proc_audio = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    )

    batch["input_values"] = proc_audio["input_values"][0]
    batch["attention_mask"] = proc_audio["attention_mask"][0]

    # Labels
    text = batch["transcription"].lower().strip()
    batch["labels"] = processor.tokenizer(text).input_ids

    return batch

In [13]:
# ── Apply preprocessing to all splits ──
cols_to_remove = [c for c in hf_dict["train"].column_names if c not in ["audio", "transcription", "speaker", "Category", "split"]]

train_p = hf_dict["train"].map(
    prepare_example,
    remove_columns=cols_to_remove,
    num_proc=1
)

val_p = hf_dict["val"].map(
    prepare_example,
    remove_columns=cols_to_remove,
    num_proc=1
)

test_p = hf_dict["test"].map(
    prepare_example,
    remove_columns=cols_to_remove,
    num_proc=1
)

print(train_p)
print(val_p)
print(test_p)

Map (num_proc=1):   0%|          | 0/5259 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/736 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/1786 [00:00<?, ? examples/s]

Dataset({
    features: ['audio', 'transcription', 'speaker', 'Category', 'split', 'input_values', 'attention_mask', 'labels'],
    num_rows: 5259
})
Dataset({
    features: ['audio', 'transcription', 'speaker', 'Category', 'split', 'input_values', 'attention_mask', 'labels'],
    num_rows: 736
})
Dataset({
    features: ['audio', 'transcription', 'speaker', 'Category', 'split', 'input_values', 'attention_mask', 'labels'],
    num_rows: 1786
})


## Step 4 — Data Collator

Pads variable-length waveforms and label sequences within each batch.
Label padding uses -100 so CTC loss ignores those positions.

In [14]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

# ── Data collator ──
@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = "longest"

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Inputs
        input_features = [
            {
                "input_values": f["input_values"],
                "attention_mask": f["attention_mask"],
            }
            for f in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # Labels
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels

        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)
print("Data collator ready.")

Data collator ready.


## Step 5 — Load WavLM-base Model

- `WavLMForCTC` = WavLM-base encoder + linear CTC head (768 → vocab_size)
- CNN feature encoder is frozen (standard practice)
- CTC head is randomly initialised → needs higher learning rate

In [15]:
# ── Model init ──
import os
from transformers import WavLMForCTC

MODEL_NAME = "microsoft/wavlm-base"

model = WavLMForCTC.from_pretrained(
    MODEL_NAME,
    cache_dir=os.environ.get("TRANSFORMERS_CACHE", None),
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ctc_zero_infinity=True,
)

# Freeze only the CNN feature extractor
model.freeze_feature_encoder()

# FIX: Enable gradient checkpointing to reduce activation memory ~40%
# Must be called before moving to device and before Trainer init
model.gradient_checkpointing_enable()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Tokenizer size:        {len(processor.tokenizer):,}")
print(f"Model vocab size:      {model.config.vocab_size:,}")
print(f"LM head out features:  {model.lm_head.out_features:,}")
print(f"Pad token id:          {processor.tokenizer.pad_token_id}")
print(f"Total parameters:      {total_params:,}")
print(f"Trainable parameters:  {trainable_params:,}")
print(f"Frozen parameters:     {frozen_params:,}")
print(f"Model on device:       {next(model.parameters()).device}")


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

WavLMForCTC LOAD REPORT from: microsoft/wavlm-base
Key            | Status  | 
---------------+---------+-
lm_head.bias   | MISSING | 
lm_head.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizer size:        32
Model vocab size:      32
LM head out features:  32
Pad token id:          29
Total parameters:      94,406,544
Trainable parameters:  90,206,096
Frozen parameters:     4,200,448
Model on device:       cpu


model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

## Step 6 — Metrics

**FIX: Proper CTC decoding in compute_metrics.**

The raw argmax output from the model contains repeated characters and blank tokens.
CTC decoding requires: (1) collapse consecutive duplicate tokens, (2) remove blank tokens.
`Wav2Vec2CTCTokenizer.batch_decode` does NOT do CTC collapse — it just maps token IDs to chars.
We must manually collapse before decoding, or use `processor.batch_decode` which handles it.

In [16]:
# ── Metrics ──
import numpy as np
import evaluate
from itertools import groupby

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def decode_ctc_predictions(pred_ids, tokenizer):
    blank_id = tokenizer.pad_token_id
    decoded = []

    for seq in pred_ids:
        collapsed = [k for k, _ in groupby(seq)]
        filtered = [t for t in collapsed if t != blank_id]

        if len(filtered) == 0:
            decoded.append("")
        else:
            decoded.append(tokenizer.decode(filtered, skip_special_tokens=True))

    return decoded

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    label_ids = pred.label_ids

    # Replace -100 with pad token for decoding
    label_ids = np.where(
        label_ids != -100,
        label_ids,
        processor.tokenizer.pad_token_id
    )

    pred_str = decode_ctc_predictions(pred_ids, processor.tokenizer)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str = [p.strip() for p in pred_str]
    label_str = [l.strip() for l in label_str]

    # Avoid empty-list issues in debug metrics
    pred_eval = [p if len(p) > 0 else "⟨empty⟩" for p in pred_str]
    label_eval = [l if len(l) > 0 else "⟨empty⟩" for l in label_str]

    wer = wer_metric.compute(predictions=pred_eval, references=label_eval)
    cer = cer_metric.compute(predictions=pred_eval, references=label_eval)

    return {"wer": wer, "cer": cer}

## Step 7 — Training Arguments & Optimizer

**Key fixes:**
- **Layered learning rates**: CTC head gets 1e-3 (it's randomly initialised),
  pretrained WavLM encoder gets 3e-5 (gentle fine-tuning)
- **40 epochs** instead of 10 — CTC alignment learning has a delayed "hockey stick" curve
- **Effective batch size 16** (8 × 2) instead of 32 — more frequent updates on small data
- **Warmup ratio 0.15** — longer warmup helps CTC stability
- **Early stopping patience 8** — CTC needs time before metrics start dropping

In [17]:
# ── Training arguments ──
from transformers import TrainingArguments

CHECKPOINT_DIR = "/kaggle/working/b2_wavlm_ctc"

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,

    # Training schedule
    num_train_epochs=40,                 # FIX: CTC needs many epochs
    per_device_train_batch_size=4,       # FIX: increased from 2 (single GPU now has full memory)
    gradient_accumulation_steps=4,       # effective batch = 16
    learning_rate=3e-4,                  # FIX: standard WavLM CTC LR (was 1e-4)
    warmup_ratio=0.15,                   # FIX: longer warmup (was 0.03)
    lr_scheduler_type="linear",
    weight_decay=0.005,

    # Eval + checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,

    # Memory
    fp16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=2,

    # Logging
    logging_steps=25,
    logging_dir="/kaggle/temp/tb_logs",
    report_to="none",

    # Save space
    save_total_limit=3,
)

print("Training arguments set.")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments set.
Effective batch size: 16


In [18]:
# ── Callbacks ──
from transformers import TrainerCallback, EarlyStoppingCallback
import torch

class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        if "loss" in logs:
            print(f"[step {state.global_step}] train_loss={logs['loss']:.4f}")
        if "learning_rate" in logs:
            print(f"[step {state.global_step}] lr={logs['learning_rate']:.2e}")
        if "eval_cer" in logs:
            print(f"[step {state.global_step}] val_cer={logs['eval_cer']:.4f}  val_wer={logs.get('eval_wer', 'N/A')}")

class SanityCheckCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return

        model.eval()
        device = next(model.parameters()).device

        try:
            sample = val_p[0]
            input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)
            attention_mask = torch.tensor(sample["attention_mask"]).unsqueeze(0).to(device)

            with torch.no_grad():
                logits = model(
                    input_values=input_values,
                    attention_mask=attention_mask
                ).logits

            pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()
            pred_text = decode_ctc_predictions(pred_ids, processor.tokenizer)[0]

            label_ids = [x for x in sample["labels"] if x != -100]
            ref_text = processor.tokenizer.decode(label_ids, skip_special_tokens=True)

            print(f"  [SANITY CHECK] REF : {ref_text}")
            print(f"  [SANITY CHECK] PRED: {pred_text}")

        except Exception as e:
            print(f"  [SANITY CHECK] Error: {e}")

# FIX: patience 8, not 2 — CTC has delayed convergence
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=8,
    early_stopping_threshold=0.001,
)

print("Callbacks ready.")

Callbacks ready.


## Step 8 — Train

**FIX v2: Simplified optimizer — let Trainer handle it.**

The custom optimizer with layered learning rates caused a scheduling bug where the
CTC head LR decayed to near-zero before learning anything (model predicted all blanks).

Now using a single LR of 3e-4, which is the standard for wav2vec2/WavLM CTC fine-tuning.
The CTC head naturally receives larger gradients (it's randomly initialized), so it
effectively gets a higher update magnitude even with a shared LR. The Trainer handles
warmup and linear decay correctly.

In [19]:
# ── Trainer ──
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_p,
    eval_dataset=val_p,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[PrintLossCallback(), SanityCheckCallback(), early_stopping],
)

print("\n--- DISK USAGE BEFORE TRAINING ---")
!du -sh /kaggle/working/* 2>/dev/null || true
!du -sh /kaggle/temp/* 2>/dev/null || true

print("\nStarting training...")
trainer.train()


--- DISK USAGE BEFORE TRAINING ---
4.0K	/kaggle/working/b2_wavlm_ctc
796M	/kaggle/working/torgo_b2_split
4.0K	/kaggle/working/vocab.json
20K	/kaggle/working/wavlm_processor
7.4G	/kaggle/temp/hf

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Epoch,Training Loss,Validation Loss,Wer,Cer
1,14.229764,3.397442,1.000000,1.091964
2,11.806525,2.895030,1.000000,1.092302
3,7.471163,1.560755,0.889074,0.486194
4,6.009720,1.585647,0.790904,0.385101
5,5.039671,1.345164,0.713810,0.385777
6,5.231672,1.686666,0.696062,0.336865
7,4.609539,1.141137,0.605103,0.258988
8,4.150457,1.049214,0.569052,0.254367
9,3.524583,1.020444,0.498059,0.213682
10,3.017619,0.806974,0.434831,0.190015


[step 25] train_loss=295.5970
[step 25] lr=3.65e-06
[step 50] train_loss=235.9882
[step 50] lr=7.45e-06
[step 75] train_loss=98.2610
[step 75] lr=1.12e-05
[step 100] train_loss=42.3715
[step 100] lr=1.50e-05
[step 125] train_loss=24.3399
[step 125] lr=1.88e-05
[step 150] train_loss=22.3025
[step 150] lr=2.26e-05
[step 175] train_loss=21.6464
[step 175] lr=2.64e-05
[step 200] train_loss=17.5778
[step 200] lr=3.02e-05
[step 225] train_loss=17.2102
[step 225] lr=3.40e-05
[step 250] train_loss=15.1208
[step 250] lr=3.78e-05
[step 275] train_loss=14.8587
[step 275] lr=4.16e-05
[step 300] train_loss=14.0986
[step 300] lr=4.54e-05
[step 325] train_loss=14.2298
[step 325] lr=4.92e-05
[step 329] val_cer=1.0920  val_wer=1.0
  [SANITY CHECK] REF : air
  [SANITY CHECK] PRED: 


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


...

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


[step 12850] train_loss=0.5131
[step 12850] lr=8.34e-06
[step 12875] train_loss=0.4645
[step 12875] lr=7.67e-06
[step 12900] train_loss=0.6172
[step 12900] lr=7.00e-06
[step 12925] train_loss=0.3931
[step 12925] lr=6.33e-06
[step 12950] train_loss=0.4715
[step 12950] lr=5.66e-06
[step 12975] train_loss=0.4224
[step 12975] lr=4.99e-06
[step 13000] train_loss=0.3824
[step 13000] lr=4.32e-06
[step 13025] train_loss=0.4946
[step 13025] lr=3.65e-06
[step 13050] train_loss=0.4324
[step 13050] lr=2.98e-06
[step 13075] train_loss=0.3492
[step 13075] lr=2.31e-06
[step 13100] train_loss=0.3971
[step 13100] lr=1.64e-06
[step 13125] train_loss=0.4479
[step 13125] lr=9.65e-07
[step 13150] train_loss=0.3913
[step 13150] lr=2.95e-07
[step 13160] val_cer=0.0559  val_wer=0.12534664448141986
  [SANITY CHECK] REF : air
  [SANITY CHECK] PRED: air


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=13160, training_loss=3.964658372144931, metrics={'train_runtime': 11555.7832, 'train_samples_per_second': 18.204, 'train_steps_per_second': 1.139, 'total_flos': 9.974582212086194e+18, 'train_loss': 3.964658372144931, 'epoch': 40.0})

In [20]:
# ── Save the final best model ──
FINAL_MODEL_PATH = "/kaggle/working/b2_wavlm_ctc_final"
trainer.save_model(FINAL_MODEL_PATH)
processor.save_pretrained(FINAL_MODEL_PATH)

print("Best model saved to:", FINAL_MODEL_PATH)

print("\n--- DISK USAGE AFTER TRAINING ---")
!du -sh /kaggle/working/* 2>/dev/null || true

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved to: /kaggle/working/b2_wavlm_ctc_final

--- DISK USAGE AFTER TRAINING ---
3.1G	/kaggle/working/b2_wavlm_ctc
361M	/kaggle/working/b2_wavlm_ctc_final
796M	/kaggle/working/torgo_b2_split
4.0K	/kaggle/working/vocab.json
20K	/kaggle/working/wavlm_processor


## Step 9 — Evaluation on Test Set

Report overall WER/CER and per-severity breakdown.
**FIX: Uses proper CTC decoding (collapse + blank removal).**

In [23]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = "longest"

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Inputs — only pass input_values, let pad() generate attention_mask
        input_features = [
            {"input_values": f["input_values"]}
            for f in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # Labels
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels

        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

In [24]:
# ── Greedy CTC decoding on test set ──
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

all_preds  = []
all_labels = []
all_cats   = []

TEST_BATCH_SIZE = 8

for i in range(0, len(test_p), TEST_BATCH_SIZE):
    end = min(i + TEST_BATCH_SIZE, len(test_p))

    batch = data_collator([
        {
            "input_values": test_p[j]["input_values"],
            "labels":       test_p[j]["labels"],
        }
        for j in range(i, end)
    ])

    input_values  = batch["input_values"].to(device)
    attention_mask = batch["attention_mask"].to(device) if "attention_mask" in batch else None

    with torch.no_grad():
        logits = model(
            input_values,
            attention_mask=attention_mask
        ).logits

    pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()

    # Proper CTC decoding
    pred_str = decode_ctc_predictions(pred_ids, processor.tokenizer)

    # Decode labels
    label_ids = batch["labels"].numpy()
    label_ids = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    all_preds.extend([p.strip() for p in pred_str])
    all_labels.extend([l.strip() for l in label_str])
    all_cats.extend([test_p[j]["Category"] for j in range(i, end)])

print(f"Decoded {len(all_preds)} test utterances.")

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding

Decoded 1786 test utterances.


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


In [25]:
# ── Overall metrics ──
# Filter empty references
valid = [(p, l, c) for p, l, c in zip(all_preds, all_labels, all_cats) if len(l.strip()) > 0]
v_preds = [p if len(p) > 0 else "⟨empty⟩" for p, _, _ in valid]
v_labels = [l for _, l, _ in valid]
v_cats = [c for _, _, c in valid]

overall_wer = wer_metric.compute(predictions=v_preds, references=v_labels)
overall_cer = cer_metric.compute(predictions=v_preds, references=v_labels)

print("=" * 50)
print("B2 — WavLM-base + CTC — TEST RESULTS")
print("=" * 50)
print(f"Overall WER: {overall_wer:.4f}  ({overall_wer*100:.2f}%)")
print(f"Overall CER: {overall_cer:.4f}  ({overall_cer*100:.2f}%)")
print()

B2 — WavLM-base + CTC — TEST RESULTS
Overall WER: 0.3075  (30.75%)
Overall CER: 0.1694  (16.94%)



In [26]:
# ── Per-severity breakdown ──
results_df = pd.DataFrame({
    "prediction": all_preds,
    "reference":  all_labels,
    "Category":   all_cats,
})

print("Per-severity results:")
print("-" * 50)

for cat in ["control", "mild", "moderate", "severe"]:
    subset = results_df[results_df["Category"] == cat]
    subset = subset[subset["reference"].str.strip().str.len() > 0]
    if len(subset) == 0:
        print(f"{cat:15s}: no samples")
        continue

    preds = [p if len(p) > 0 else "⟨empty⟩" for p in subset["prediction"].tolist()]
    cat_wer = wer_metric.compute(predictions=preds, references=subset["reference"].tolist())
    cat_cer = cer_metric.compute(predictions=preds, references=subset["reference"].tolist())
    print(f"{cat:15s}: WER={cat_wer*100:.2f}%  CER={cat_cer*100:.2f}%  (n={len(subset)})")

print()

results_df.to_csv("/kaggle/working/b2_test_results.csv", index=False)
print("Results saved to /kaggle/working/b2_test_results.csv")

summary = {
    "model": "B2_WavLM_CTC",
    "overall_wer": round(overall_wer, 4),
    "overall_cer": round(overall_cer, 4),
    "test_speakers": ["F01", "F04", "FC01", "M05"],
}
with open("/kaggle/working/b2_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Summary saved to /kaggle/working/b2_summary.json")

Per-severity results:
--------------------------------------------------
control        : WER=7.79%  CER=4.09%  (n=302)
mild           : WER=9.61%  CER=3.98%  (n=673)
moderate       : WER=60.58%  CER=34.93%  (n=575)
severe         : WER=54.61%  CER=32.16%  (n=236)

Results saved to /kaggle/working/b2_test_results.csv
Summary saved to /kaggle/working/b2_summary.json


In [27]:
# ── Sample predictions ──
print("\nSample predictions (first 15 test utterances):")
print("-" * 70)
for i in range(min(15, len(all_preds))):
    print(f"[{all_cats[i]:10s}] REF : {all_labels[i]}")
    print(f"           PRED: {all_preds[i]}")
    print()


Sample predictions (first 15 test utterances):
----------------------------------------------------------------------
[control   ] REF : alpha
           PRED: alpha

[control   ] REF : the
           PRED: the

[control   ] REF : except in the winter when the oze or snow or ice prevents
           PRED: except in the winter when the oze or snow or ice prevents

[control   ] REF : raid
           PRED: raid

[control   ] REF : read
           PRED: lead

[control   ] REF : stuble
           PRED: stuble

[control   ] REF : ate
           PRED: ake

[control   ] REF : store
           PRED: storm

[control   ] REF : sip
           PRED: sip

[control   ] REF : wish
           PRED: wish

[control   ] REF : slay
           PRED: slay

[control   ] REF : sigh
           PRED: sigh

[control   ] REF : jaged
           PRED: jaged

[control   ] REF : up
           PRED: up

[control   ] REF : chair
           PRED: chair



In [28]:
!zip -r /kaggle/working/b2_wavlm_ctc_final.zip /kaggle/working/b2_wavlm_ctc_final/

  adding: kaggle/working/b2_wavlm_ctc_final/ (stored 0%)
  adding: kaggle/working/b2_wavlm_ctc_final/added_tokens.json (deflated 20%)
  adding: kaggle/working/b2_wavlm_ctc_final/config.json (deflated 66%)
  adding: kaggle/working/b2_wavlm_ctc_final/vocab.json (deflated 56%)
  adding: kaggle/working/b2_wavlm_ctc_final/processor_config.json (deflated 44%)
  adding: kaggle/working/b2_wavlm_ctc_final/tokenizer_config.json (deflated 73%)
  adding: kaggle/working/b2_wavlm_ctc_final/model.safetensors (deflated 9%)
  adding: kaggle/working/b2_wavlm_ctc_final/training_args.bin (deflated 53%)
